# Day 1: Data Curation — Vietnamese Price Prediction Dataset (v4)

Pipeline: Load JSONL files -> Clean (9-step Vietnamese) -> Category mapping (8 categories) -> Dedup -> EDA -> Weighted sampling -> Split -> Push HF Hub

**Data sources:** Tiki Scraper (102K) + Kaggle (42K) + Hasaki (11K) + WinMart (3.2K) = ~158K raw

**Config:** TRAIN_SIZE=110K, VAL=5K, TEST=5K, total=120K

**Penalties:** Thoi Trang 0.40, Nha Cua 0.60

**HuggingFace:** `SeanSunny/items_raw_tv_v4`

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

from pricer_vi.items import Item
from pricer_vi.parser import parse

In [ ]:
# --- Configuration ---

SCRIPT_DIR = Path(".").resolve()
DATA_DIR = SCRIPT_DIR.parent / "Tiki" / "Tiki_dataset_scrape"
OUTPUT_DIR = SCRIPT_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
HF_DATASET_NAME = "SeanSunny/items_raw_tv_v4"
TRAIN_SIZE = 110_000
VAL_SIZE = 5_000
TEST_SIZE = 5_000

print(f"Data dir: {DATA_DIR}")
print(f"Total target: {TRAIN_SIZE + VAL_SIZE + TEST_SIZE:,} items")

In [ ]:
# --- 8 Target Categories + Mapping ---

CATEGORY_MAP = {
    # Thoi Trang (gop fashion + phu kien + balo)
    "Th\u1eddi Trang": "Th\u1eddi Trang",
    "Th\u1eddi trang n\u1eef": "Th\u1eddi Trang",
    "Th\u1eddi trang nam": "Th\u1eddi Trang",
    "Ph\u1ee5 ki\u1ec7n th\u1eddi trang": "Th\u1eddi Trang",
    "Balo v\u00e0 Vali": "Th\u1eddi Trang",
    # Nha Cua (gop cham soc nha cua)
    "Nh\u00e0 C\u1eeda - \u0110\u1eddi S\u1ed1ng": "Nh\u00e0 C\u1eeda - \u0110\u1eddi S\u1ed1ng",
    "Ch\u0103m s\u00f3c nh\u00e0 c\u1eeda": "Nh\u00e0 C\u1eeda - \u0110\u1eddi S\u1ed1ng",
    # Dien Tu - Cong Nghe
    "Laptop - M\u00e1y Vi T\u00ednh - Linh ki\u1ec7n": "\u0110i\u1ec7n T\u1eed - C\u00f4ng Ngh\u1ec7",
    "Thi\u1ebft B\u1ecb S\u1ed1 - Ph\u1ee5 Ki\u1ec7n S\u1ed1": "\u0110i\u1ec7n T\u1eed - C\u00f4ng Ngh\u1ec7",
    # Lam Dep
    "L\u00e0m \u0110\u1eb9p - S\u1ee9c Kh\u1ecfe": "L\u00e0m \u0110\u1eb9p - S\u1ee9c Kh\u1ecfe",
    # Me va Be (gop Do Choi)
    "\u0110\u1ed3 Ch\u01a1i - M\u1eb9 & B\u00e9": "M\u1eb9 v\u00e0 B\u00e9",
    # Dien Lanh va Gia Dung
    "\u0110i\u1ec7n Gia D\u1ee5ng": "\u0110i\u1ec7n L\u1ea1nh v\u00e0 Gia D\u1ee5ng",
    "\u0110i\u1ec7n T\u1eed - \u0110i\u1ec7n L\u1ea1nh": "\u0110i\u1ec7n L\u1ea1nh v\u00e0 Gia D\u1ee5ng",
    # Bach Hoa
    "B\u00e1ch H\u00f3a Online": "B\u00e1ch H\u00f3a",
    "Th\u1ef1c ph\u1ea9m": "B\u00e1ch H\u00f3a",
    "Phi th\u1ef1c ph\u1ea9m": "B\u00e1ch H\u00f3a",
    # O To
    "\u00d4 T\u00f4 - Xe M\u00e1y - Xe \u0110\u1ea1p": "\u00d4 T\u00f4 - Xe M\u00e1y",
    # Bo
    "Nh\u00e0 S\u00e1ch Tiki": None,
}

SUBCATEGORY_OVERRIDES = {
    "Th\u1ec3 thao": "Nh\u00e0 C\u1eeda - \u0110\u1eddi S\u1ed1ng",
    "Qu\u00e0 l\u01b0u ni\u1ec7m": "M\u1eb9 v\u00e0 B\u00e9",
    "V\u0103n ph\u00f2ng ph\u1ea9m": "Nh\u00e0 C\u1eeda - \u0110\u1eddi S\u1ed1ng",
}

CATEGORY_PENALTIES = {
    "Th\u1eddi Trang": 0.40,
    "Nh\u00e0 C\u1eeda - \u0110\u1eddi S\u1ed1ng": 0.60,
}

print(f"Categories: {len(set(v for v in CATEGORY_MAP.values() if v)):}")
print(f"Penalties: {CATEGORY_PENALTIES}")

## Step 1: Explore a single JSONL file

Before loading everything, let's look at what one file contains.

In [ ]:
# Pick one file to explore
sample_files = sorted(DATA_DIR.glob("*.jsonl"))[:5]
for f in sample_files:
    count = sum(1 for _ in open(f, encoding="utf-8"))
    print(f"{f.name}: {count:,} items")

In [ ]:
# Look at one raw datapoint
first_file = sorted(DATA_DIR.glob("*.jsonl"))[0]
with open(first_file, encoding="utf-8") as f:
    datapoint = json.loads(f.readline())

print(f"File: {first_file.name}")
print(f"Keys: {list(datapoint.keys())}")
print(f"Title: {datapoint.get('title', '')[:100]}")
print(f"Price: {datapoint.get('price', 0):,} VND")
print(f"Category: {datapoint.get('category', '')}")
print(f"Brand: {datapoint.get('brand', '')}")
print(f"Features (first 300 chars): {datapoint.get('features', '')[:300]}")

In [ ]:
# Parse it into an Item
raw_cat = datapoint.get("category", "")
parent = raw_cat.split(" > ")[0]
mapped = CATEGORY_MAP.get(parent, "UNKNOWN")
print(f"Raw category: {raw_cat}")
print(f"Parent: {parent}")
print(f"Mapped to: {mapped}")

if mapped and mapped != "UNKNOWN":
    item = parse(datapoint, mapped)
    if item:
        print(f"\nParsed: {item}")
        print(f"Full text ({len(item.full)} chars):\n{item.full[:500]}")

In [ ]:
# Explore a WinMart file (FMCG)
winmart_files = sorted(DATA_DIR.glob("winmart_*.jsonl"))
print(f"WinMart files: {len(winmart_files)}")
for f in winmart_files[:3]:
    count = sum(1 for _ in open(f, encoding="utf-8"))
    print(f"  {f.name}: {count:,} items")

if winmart_files:
    with open(winmart_files[0], encoding="utf-8") as f:
        wm = json.loads(f.readline())
    print(f"\nWinMart sample:")
    print(f"  Title: {wm.get('title', '')}")
    print(f"  Price: {wm.get('price', 0):,} VND")
    print(f"  Category: {wm.get('category', '')}")
    print(f"  Features: {wm.get('features', '')[:200]}")

## Step 2: Load all JSONL files

Load 263 JSONL files from Tiki + Kaggle + Hasaki + WinMart, apply category mapping.

In [ ]:
def _map_category(raw_cat: str, is_kaggle: bool) -> str | None:
    """Map a raw category string to one of 8 target categories, or None to drop."""
    if is_kaggle:
        return "Th\u1eddi Trang"
    parts = raw_cat.split(" > ")
    parent = parts[0]
    sub = parts[1] if len(parts) > 1 else ""
    for keyword, target in SUBCATEGORY_OVERRIDES.items():
        if keyword in sub or keyword in parent:
            return target
    if parent in CATEGORY_MAP:
        return CATEGORY_MAP[parent]
    return None


def load_all_items() -> list[Item]:
    """Load all JSONL files, parse into Item objects with 8-category mapping."""
    items = []
    dropped_cats = Counter()
    for filepath in tqdm(sorted(DATA_DIR.glob("*.jsonl")), desc="Loading files"):
        is_kaggle = "kaggle" in filepath.name
        count = 0
        for line in open(filepath, encoding="utf-8"):
            datapoint = json.loads(line)
            raw_cat = datapoint.get("category", "")
            category = _map_category(raw_cat, is_kaggle)
            if category is None:
                dropped_cats[raw_cat.split(" > ")[0] if " > " in raw_cat else raw_cat] += 1
                continue
            item = parse(datapoint, category)
            if item:
                items.append(item)
                count += 1
        if count > 0:
            print(f"  {filepath.name}: {count:,} items")
    if dropped_cats:
        print(f"\nDropped categories:")
        for cat, cnt in dropped_cats.most_common():
            print(f"  {cat}: {cnt:,} items")
    return items

In [ ]:
items = load_all_items()
print(f"\nLoaded: {len(items):,} items from {len(list(DATA_DIR.glob('*.jsonl')))} files")

In [ ]:
# Quick look at a few items
for item in items[:3]:
    print(f"{item}  |  full={len(item.full)} chars  |  brand={item.brand}")
    print(f"  {item.full[:150]}...")
    print()

## Step 3: EDA before dedup

Explore price, text length, and category distributions on the raw loaded data.

In [ ]:
def plot_eda(items: list[Item], prefix: str = "", save: bool = True):
    """Generate 5 EDA charts. Optionally save to output/."""
    lengths = [len(item.full) for item in items]
    prices = [item.price for item in items]
    cat_counts = Counter([item.category for item in items])
    categories = list(cat_counts.keys())
    counts = [cat_counts[c] for c in categories]

    # --- Stats ---
    print(f"--- EDA Stats ({prefix.strip('_') or 'raw'}) ---")
    print(f"Total items: {len(items):,}")
    print(f"Price: min={min(prices):,}, max={max(prices):,}, avg={sum(prices)/len(prices):,.0f}, median={int(np.median(prices)):,}")
    print(f"Text length: min={min(lengths)}, max={max(lengths):,}, avg={sum(lengths)/len(lengths):,.0f}")
    print(f"Categories: {len(cat_counts)}")
    for cat, cnt in cat_counts.most_common():
        print(f"  {cat}: {cnt:,} ({cnt/len(items)*100:.1f}%)")

    # 1. Text length histogram
    plt.figure(figsize=(15, 6))
    plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
    plt.xlabel("Length (chars)")
    plt.ylabel("Count")
    plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, min(max(lengths) + 100, 6000), 100))
    if save:
        plt.savefig(OUTPUT_DIR / f"{prefix}length_distribution.png", dpi=100, bbox_inches="tight")
    plt.show()

    # 2. Price histogram (log-scale for VND)
    plt.figure(figsize=(15, 6))
    plt.title(f"Prices (VND): Avg {sum(prices)/len(prices):,.0f}, Median {int(np.median(prices)):,}\n")
    plt.xlabel("Price (VND)")
    plt.ylabel("Count")
    log_bins = np.logspace(np.log10(max(min(prices), 1)), np.log10(max(prices) + 1), 50)
    plt.hist(prices, rwidth=0.7, color="blueviolet", bins=log_bins)
    plt.xscale("log")
    if save:
        plt.savefig(OUTPUT_DIR / f"{prefix}price_distribution.png", dpi=100, bbox_inches="tight")
    plt.show()

    # 3. Category bar chart
    plt.figure(figsize=(15, 6))
    plt.bar(categories, counts, color="goldenrod")
    plt.title("How many in each category")
    plt.xlabel("Categories")
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    for i, v in enumerate(counts):
        plt.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    if save:
        plt.savefig(OUTPUT_DIR / f"{prefix}category_distribution.png", dpi=100, bbox_inches="tight")
    plt.show()

    # 4. Category pie chart
    plt.figure(figsize=(12, 10))
    plt.pie(counts, labels=categories, autopct="%1.0f%%", startangle=90)
    centre_circle = plt.Circle((0, 0), 0.70, fc="white")
    plt.gcf().gca().add_artist(centre_circle)
    plt.title("Categories")
    plt.axis("equal")
    if save:
        plt.savefig(OUTPUT_DIR / f"{prefix}category_pie.png", dpi=100, bbox_inches="tight")
    plt.show()

    # 5. Price vs text length scatter
    plt.figure(figsize=(15, 8))
    plt.scatter(lengths, prices, s=0.2, color="red")
    plt.xlabel("Text length (chars)")
    plt.ylabel("Price (VND)")
    plt.title("Is there a simple correlation with text length?")
    if save:
        plt.savefig(OUTPUT_DIR / f"{prefix}price_vs_length.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
plot_eda(items, prefix="01_before_dedup_")

## Step 4: Deduplication

Shuffle then remove duplicates by title and by full text to prevent data leakage.

In [ ]:
random.seed(RANDOM_SEED)
random.shuffle(items)

before = len(items)

seen = set()
items = [x for x in tqdm(items, desc="Dedup by title")
         if not (x.title in seen or seen.add(x.title))]
after_title = len(items)

seen = set()
items = [x for x in tqdm(items, desc="Dedup by full")
         if not (x.full in seen or seen.add(x.full))]
del seen

print(f"Before: {before:,}")
print(f"After title dedup: {after_title:,} (removed {before - after_title:,})")
print(f"After full dedup: {len(items):,} (removed {after_title - len(items):,})")
print(f"Total removed: {before - len(items):,} ({(before - len(items))/before*100:.1f}%)")

## Step 5: EDA after dedup

In [ ]:
plot_eda(items, prefix="02_after_dedup_")

## Step 6: Weighted Sampling

Apply price^2 weighting (boost expensive items) and category penalties to create a balanced sample.

Same technique as English pipeline (day1.ipynb):
- `price^2` weighting shifts avg price up, reducing bias toward cheap items
- Category penalties reduce over-represented categories

In [ ]:
target_size = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
total = len(items)

print(f"Available: {total:,} items")
print(f"Target: {target_size:,} items")
print(f"Sampling rate: {target_size/total*100:.1f}%")

if total <= target_size:
    print(f"Not enough data! Using all {total:,} items.")
    sample_size = total
else:
    sample_size = target_size
    print(f"Will drop {total - target_size:,} items ({(total - target_size)/total*100:.1f}%)")

In [ ]:
np.random.seed(RANDOM_SEED)

prices = np.array([it.price for it in items], dtype=float)
categories = np.array([it.category for it in items])

# Normalize prices to [0, 1]
p = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)

# price^2 weighting — boost expensive items
w = p ** 2

# Category penalties
for cat, penalty in CATEGORY_PENALTIES.items():
    mask = categories == cat
    w[mask] *= penalty
    print(f"Penalty {cat}: {penalty} (affects {mask.sum():,} items)")

# Normalize to probability distribution
w = w / w.sum()

# Sample
idx = np.random.choice(len(items), size=sample_size, replace=False, p=w)
sample = [items[i] for i in idx]
print(f"\nSample size: {len(sample):,}")

In [ ]:
# Final shuffle
random.seed(RANDOM_SEED)
random.shuffle(sample)

## Step 7: EDA after sampling

Check that penalties worked: Thoi Trang should drop from ~36% to ~25%, price avg should increase.

In [ ]:
plot_eda(sample, prefix="03_after_sampling_")

In [ ]:
# Compare before/after sampling
before_cats = Counter([it.category for it in items])
after_cats = Counter([it.category for it in sample])

print(f"{'Category':<30} {'Before dedup':>15} {'After sampling':>15} {'Change':>10}")
print("-" * 75)
for cat in sorted(before_cats.keys(), key=lambda c: before_cats[c], reverse=True):
    b = before_cats[cat]
    a = after_cats.get(cat, 0)
    pct_b = b / len(items) * 100
    pct_a = a / len(sample) * 100
    print(f"{cat:<30} {b:>10,} ({pct_b:4.1f}%) {a:>10,} ({pct_a:4.1f}%) {pct_a - pct_b:>+7.1f}%")

## Step 8: Split dataset

Split into train / validation / test.

In [ ]:
test = sample[:TEST_SIZE]
val = sample[TEST_SIZE:TEST_SIZE + VAL_SIZE]
train = sample[TEST_SIZE + VAL_SIZE:]

print(f"Train: {len(train):,}")
print(f"Val:   {len(val):,}")
print(f"Test:  {len(test):,}")
print(f"Total: {len(train) + len(val) + len(test):,}")

In [ ]:
# Sanity check: no data leakage between splits
train_titles = set(it.title for it in train)
val_titles = set(it.title for it in val)
test_titles = set(it.title for it in test)

leak_tv = train_titles & val_titles
leak_tt = train_titles & test_titles
leak_vt = val_titles & test_titles

print(f"Train-Val overlap: {len(leak_tv)} titles")
print(f"Train-Test overlap: {len(leak_tt)} titles")
print(f"Val-Test overlap: {len(leak_vt)} titles")

if len(leak_tv) + len(leak_tt) + len(leak_vt) == 0:
    print("No data leakage detected.")
else:
    print("WARNING: Data leakage detected!")

## Step 9: Push to HuggingFace Hub

Push the final dataset as `SeanSunny/items_raw_tv_v4`.

In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("Logged in to HuggingFace.")
else:
    print("HF_TOKEN not found in .env. Set it to push.")

In [ ]:
# Push to Hub
print(f"Pushing to {HF_DATASET_NAME}...")
Item.push_to_hub(HF_DATASET_NAME, train, val, test)
print(f"Done! Dataset: https://huggingface.co/datasets/{HF_DATASET_NAME}")

## Summary

| Step | Result |
|---|---|
| Raw JSONL files | 263 files, ~158K items |
| After parse + category map | ~157K items (8 categories) |
| After dedup | ~147K items |
| Weighted sampling | 120K items (price^2 + penalties) |
| Split | 110K train / 5K val / 5K test |
| HuggingFace | `SeanSunny/items_raw_tv_v4` |